# Validación de Teleco_clean.csv antes de carga a PostgreSQL

## 1. Importar librerías

In [1]:
import pandas as pd
import logging
import os

os.makedirs('logs', exist_ok=True)
logging.basicConfig(
    filename='logs/validacion.log',
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

CSV_PATH = '../data/processed/Telco_clean.csv'
errores  = []
warnings = []

print('Librerías cargadas')

Librerías cargadas


## 2. Leer CSV

In [2]:
try:
    df = pd.read_csv(CSV_PATH)
    total_inicial = len(df)
    print(f'Filas: {len(df):,}  |  Columnas: {len(df.columns)}')
    logging.info(f'CSV leído: {len(df)} filas')
    df.head(3)
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo '{CSV_PATH}'.")
    logging.error('ERROR: no se encontro el archivo')


Filas: 7,043  |  Columnas: 19


## 3. Validación: columnas esperadas

In [3]:
COLUMNAS_ESPERADAS = [
    'gender', 'senior_citizen', 'partner', 'dependents',
    'tenure', 'phone_service', 'multiple_lines', 'internet_service',
    'online_security', 'online_backup', 'device_protection', 'tech_support',
    'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
    'payment_method', 'monthly_charges', 'churn'
]

faltantes = set(COLUMNAS_ESPERADAS) - set(df.columns)
extra     = set(df.columns) - set(COLUMNAS_ESPERADAS)

if faltantes:
    msg = f'Columnas faltantes: {faltantes}'
    errores.append(msg)
    logging.error(msg)
    print(f'ERROR: {msg}')
else:
    print('OK - Todas las columnas esperadas están presentes')

if extra:
    msg = f'Columnas extra no esperadas: {extra}'
    warnings.append(msg)
    logging.warning(msg)
    print(f'WARNING: {msg}')

if len(df) == 0:
    msg = 'El archivo está vacío'
    errores.append(msg)
    logging.error(msg)
    print(f'ERROR: {msg}')
else:
    print(f'OK - El archivo tiene {len(df):,} filas')

OK - Todas las columnas esperadas están presentes
OK - El archivo tiene 7,043 filas


## 4. Validación: valores nulos

In [4]:
# Estas columnas NO pueden tener nulos
COLS_NO_NULAS = [
    'gender', 'senior_citizen', 'partner', 'dependents',
    'tenure', 'phone_service', 'multiple_lines', 'internet_service',
    'online_security', 'online_backup', 'device_protection', 'tech_support',
    'streaming_tv', 'streaming_movies', 'contract', 'paperless_billing',
    'payment_method', 'monthly_charges', 'churn'
]
# total_charges puede ser nulo (clientes con tenure=0)

for col in COLS_NO_NULAS:
    if col not in df.columns:
        continue
    nulos = df[col].isna().sum()
    if nulos > 0:
        msg = f'{col}: {nulos} valores nulos'
        errores.append(msg)
        logging.error(msg)
        print(f'ERROR: {msg}')
    else:
        print(f'OK - {col}: sin nulos')

OK - gender: sin nulos
OK - senior_citizen: sin nulos
OK - partner: sin nulos
OK - dependents: sin nulos
OK - tenure: sin nulos
OK - phone_service: sin nulos
OK - multiple_lines: sin nulos
OK - internet_service: sin nulos
OK - online_security: sin nulos
OK - online_backup: sin nulos
OK - device_protection: sin nulos
OK - tech_support: sin nulos
OK - streaming_tv: sin nulos
OK - streaming_movies: sin nulos
OK - contract: sin nulos
OK - paperless_billing: sin nulos
OK - payment_method: sin nulos
OK - monthly_charges: sin nulos
OK - churn: sin nulos


## 5. Validación: valores categóricos (FK)

In [5]:
CATEGORICAS = {
    # columnas booleanas: solo 0 y 1
    'gender':           {0, 1},
    'senior_citizen':   {0, 1},
    'partner':          {0, 1},
    'dependents':       {0, 1},
    'phone_service':    {0, 1},
    'paperless_billing':{0, 1},
    'churn':            {0, 1},
    # columnas con 3 valores: No / Sí / Sin servicio
    'multiple_lines':   {0, 1, 2},
    'online_security':  {0, 1, 2},
    'online_backup':    {0, 1, 2},
    'device_protection':{0, 1, 2},
    'tech_support':     {0, 1, 2},
    'streaming_tv':     {0, 1, 2},
    'streaming_movies': {0, 1, 2},
    'internet_service': {0, 1, 2},
    # contrato
    'contract':         {0, 1, 2},
    # método de pago
    'payment_method':   {0, 1, 2, 3},
}

for col, valores_validos in CATEGORICAS.items():
    if col not in df.columns:
        continue
    valores_reales = set(df[col].dropna().unique())
    invalidos = valores_reales - valores_validos
    if invalidos:
        msg = f'{col}: valores inválidos encontrados: {invalidos}'
        errores.append(msg)
        logging.error(msg)
        print(f'ERROR: {msg}')
    else:
        print(f'OK - {col}: valores {sorted(valores_reales)}')

OK - gender: valores [np.int64(0), np.int64(1)]
OK - senior_citizen: valores [np.int64(0), np.int64(1)]
OK - partner: valores [np.int64(0), np.int64(1)]
OK - dependents: valores [np.int64(0), np.int64(1)]
OK - phone_service: valores [np.int64(0), np.int64(1)]
OK - paperless_billing: valores [np.int64(0), np.int64(1)]
OK - churn: valores [np.int64(0), np.int64(1)]
OK - multiple_lines: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - online_security: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - online_backup: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - device_protection: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - tech_support: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - streaming_tv: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - streaming_movies: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - internet_service: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - contract: valores [np.int64(0), np.int64(1), np.int64(2)]
OK - payment_met

## 6. Validación: consistencia lógica

In [6]:
cols_ok = set(df.columns)

# Si no tiene internet (0), los servicios de internet deben ser 2 (sin servicio)
if {'internet_service','online_security','online_backup','device_protection',
    'tech_support','streaming_tv','streaming_movies'}.issubset(cols_ok):
    sin_internet = df[df['internet_service'] == 0]
    cols_internet = ['online_security','online_backup','device_protection',
                     'tech_support','streaming_tv','streaming_movies']
    for col in cols_internet:
        inconsistentes = sin_internet[sin_internet[col] != 2]
        if len(inconsistentes) > 0:
            msg = f'{col}: {len(inconsistentes)} clientes sin internet con valor != 2'
            warnings.append(msg)
            logging.warning(msg)
            print(f'WARNING: {msg}')
        else:
            print(f'OK - {col}: consistente con internet_service=0')

# Si no tiene phone_service (0), multiple_lines debe ser 2 (sin servicio)
if {'phone_service','multiple_lines'}.issubset(cols_ok):
    sin_phone = df[df['phone_service'] == 0]
    inconsistentes = sin_phone[sin_phone['multiple_lines'] != 2]
    if len(inconsistentes) > 0:
        msg = f'multiple_lines: {len(inconsistentes)} clientes sin phone_service con valor != 2'
        warnings.append(msg)
        logging.warning(msg)
        print(f'WARNING: {msg}')
    else:
        print('OK - multiple_lines: consistente con phone_service=0')

OK - online_security: consistente con internet_service=0
OK - online_backup: consistente con internet_service=0
OK - device_protection: consistente con internet_service=0
OK - tech_support: consistente con internet_service=0
OK - streaming_tv: consistente con internet_service=0
OK - streaming_movies: consistente con internet_service=0
OK - multiple_lines: consistente con phone_service=0


## 7. Resumen final

In [7]:
print('=' * 50)
print('RESUMEN DE VALIDACIÓN')
print('=' * 50)

if errores:
    print(f'\nERRORES ({len(errores)}) — NO cargar hasta resolverlos:')
    for e in errores:
        print(f'  ✗ {e}')
else:
    print('\n✓ Sin errores críticos')

if warnings:
    print(f'\nWARNINGS ({len(warnings)}) — Revisar pero no bloquean la carga:')
    for w in warnings:
        print(f'  ⚠ {w}')
else:
    print('✓ Sin warnings')

if not errores:
    print('\n→ El archivo está listo para cargar a PostgreSQL')
    logging.info('Validación completada sin errores')
else:
    print('\n→ Corregir los errores antes de ejecutar el notebook de carga')
    logging.error(f'Validación fallida: {len(errores)} errores encontrados')
    

RESUMEN DE VALIDACIÓN

✓ Sin errores críticos
✓ Sin warnings

→ El archivo está listo para cargar a PostgreSQL


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   gender             7043 non-null   int64  
 1   senior_citizen     7043 non-null   int64  
 2   partner            7043 non-null   int64  
 3   dependents         7043 non-null   int64  
 4   tenure             7043 non-null   float64
 5   phone_service      7043 non-null   int64  
 6   multiple_lines     7043 non-null   int64  
 7   internet_service   7043 non-null   int64  
 8   online_security    7043 non-null   int64  
 9   online_backup      7043 non-null   int64  
 10  device_protection  7043 non-null   int64  
 11  tech_support       7043 non-null   int64  
 12  streaming_tv       7043 non-null   int64  
 13  streaming_movies   7043 non-null   int64  
 14  contract           7043 non-null   int64  
 15  paperless_billing  7043 non-null   int64  
 16  payment_method     7043 non-null   

In [11]:
df.to_csv('../data/processed/Telco_validated.csv', index=False)

print("Dataset limpio guardado correctamente")

Dataset limpio guardado correctamente
